In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import confusion_matrix

# CONFIG

BASE_CSV_DIR = Path(r"/path/to/csv_files")  # placeholder — edit to your local per-pathology/autoencoder CSV folder

OUTPUT_DIR = Path(r"/path/to/output/analysis_error_composition")  # placeholder — edit to your desired output folder

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRICS = ["mse", "ssim", "ppw", "edge_diff_norm"]

# Which raw-value tail is the "worst reconstruction" (higher-risk) tail, per metric.
# mse/edge_diff_norm are error measures (worse = higher value) -> the "right" tail is worst.
# ssim/ppw are similarity measures (worse = lower value) -> the "left" tail is worst.
# Same convention as 05_figures_cost_benefit/cost_benefit_figures.py (tradeoff_df). Without
# this, "left"/"right" below are raw-magnitude labels only and do NOT by themselves indicate
# which tail is the higher-risk one for PPW/SSIM.
METRIC_DIRECTION = {
    "mse": "high",
    "ssim": "low",
    "ppw": "low",
    "edge_diff_norm": "high",
}
WORST_TAIL_FOR_DIRECTION = {"high": "right", "low": "left"}

# Helper Functions

def compute_percentiles(values):
    p1_low = np.percentile(values, 1)
    p5_low = np.percentile(values, 5)
    p95_high = np.percentile(values, 95)
    p99_high = np.percentile(values, 99)
    return p1_low, p5_low, p95_high, p99_high


def extract_groups(df, metric):
    values = df[metric].dropna()
    p1_low, p5_low, p95_high, p99_high = compute_percentiles(values)
    
    mean = values.mean()
    std = values.std()
    
    # Soglie SD
    left_2sd  = mean - 2 * std
    right_2sd = mean + 2 * std
    left_3sd  = mean - 3 * std
    right_3sd = mean + 3 * std

    return {
        # Percentile-based (esistenti)
        "left_0_1":   df[df[metric] <= p1_low],
        "left_1_5":   df[(df[metric] > p1_low) & (df[metric] <= p5_low)],
        "right_0_1":  df[df[metric] >= p99_high],
        "right_1_5":  df[(df[metric] < p99_high) & (df[metric] >= p95_high)],
        
        # SD-based (nuovi) — lato sinistro
        "left_2sd":   df[df[metric] <= left_2sd],
        "left_3sd":   df[df[metric] <= left_3sd],
        
        # SD-based (nuovi) — lato destro
        "right_2sd":  df[df[metric] >= right_2sd],
        "right_3sd":  df[df[metric] >= right_3sd],
    }


def compute_stats(df):
    if df.empty:
        return None

    y_true = df["true_label"]
    y_pred = df["pred_label_thr_youden"]

    if y_true.nunique() < 2:
        return None

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    total = tn + fp + fn + tp

    return {
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "FN_pct": fn / total,
        "FP_pct": fp / total,
        "Correct_pct": (tp + tn) / total,
        "Error_pct": (fp + fn) / total,
        "FN_ratio": fn / (fn + fp) if (fn + fp) > 0 else np.nan
    }


# MAIN LOOP

all_results = []

for csv_path in BASE_CSV_DIR.glob("*.csv"):

    df = pd.read_csv(csv_path, sep=";")
    df.columns = df.columns.str.strip().str.lower()

    file_tag = csv_path.stem
    pathology = file_tag.split("_")[2]
    autoencoder = file_tag.split("_")[-1]

    for metric in METRICS:

        groups = extract_groups(df, metric)

        for group_name, df_group in groups.items():

            stats = compute_stats(df_group)
            if stats is None:
                continue

            # Parsing del group_name per estrarre tail e severity
            if group_name.startswith("left_"):
                tail = "left"
                severity = group_name[len("left_"):]
            elif group_name.startswith("right_"):
                tail = "right"
                severity = group_name[len("right_"):]
            else:
                tail = "unknown"
                severity = group_name

            is_worst_tail = (tail == WORST_TAIL_FOR_DIRECTION[METRIC_DIRECTION[metric]])

            stats.update({
                "pathology": pathology,
                "autoencoder": autoencoder,
                "metric": metric,
                "tail": tail,
                "severity": severity,
                "group": group_name,
                "is_worst_reconstruction_tail": is_worst_tail
            })

            all_results.append(stats)


summary_df = pd.DataFrame(all_results)
OUTPUT_DIR.mkdir(exist_ok=True)
summary_df.to_csv(OUTPUT_DIR / "error_composition_summary.csv", index=False)

print(f"Summary saved. Total rows: {len(summary_df)}")
print(f"Groups found: {sorted(summary_df['group'].unique())}")

In [ ]:
print(summary_df["pathology"].unique())
print(summary_df["autoencoder"].unique())
print(summary_df["metric"].unique())
print(summary_df["tail"].unique())
print(summary_df["severity"].unique())
print(summary_df["group"].unique())  # aggiunto per vedere tutti i gruppi completi
print(f"Total rows: {len(summary_df)}")

In [ ]:
def build_final_long_table(summary_df):
    
    df = summary_df.copy()
    df["tail_combined"] = df["group"]
    
    totals = df["TN"] + df["TP"] + df["FP"] + df["FN"]
    
    group_order = [
        "left_0_1", "left_1_5",
        "right_0_1", "right_1_5",
        "left_2sd", "left_3sd",
        "right_2sd", "right_3sd"
    ]
    
    final_df = df[[
        "autoencoder", "tail_combined", "metric", "pathology",
        "TN", "TP", "FP", "FN", "is_worst_reconstruction_tail"
    ]].rename(columns={"tail_combined": "tail"}).copy()
    
    # Aggiungi colonne assolute in coda (prima di convertire in percentuali)
    final_df["TN_count"] = df["TN"].values
    final_df["TP_count"] = df["TP"].values
    final_df["FP_count"] = df["FP"].values
    final_df["FN_count"] = df["FN"].values
    
    # Converti le prime quattro in percentuali
    for cls in ["TN", "TP", "FP", "FN"]:
        final_df[cls] = (df[cls] / totals * 100).round(1)
    
    # Rinomina le colonne percentuali
    final_df = final_df.rename(columns={
        "TN": "TN (%)", "TP": "TP (%)",
        "FP": "FP (%)", "FN": "FN (%)"
    })
    
    # Ordine categorico
    final_df["tail"] = pd.Categorical(
        final_df["tail"], categories=group_order, ordered=True
    )
    
    final_df = final_df.sort_values(by=[
        "autoencoder", "tail", "metric", "pathology"
    ])
    
    final_df.to_csv(
        OUTPUT_DIR / "FINAL_LONG_TABLE.csv", index=False
    )

    # Companion table restricted to the worst-reconstruction tail only, per metric
    # (i.e. one row per pathology/autoencoder/metric/severity, direction-corrected) —
    # this is the subset that matches the paper's actual prioritization logic
    # (Methods, Section 2.7).
    worst_tail_df = final_df[final_df["is_worst_reconstruction_tail"]].copy()
    worst_tail_df.to_csv(
        OUTPUT_DIR / "FINAL_LONG_TABLE_worst_tail_only.csv", index=False
    )
    
    print(f"Final table saved. Rows: {len(final_df)}")
    print(f"Columns: {list(final_df.columns)}")
    print(f"Worst-tail-only table saved. Rows: {len(worst_tail_df)}")
    
    return final_df

In [ ]:
final_table = build_final_long_table(summary_df)
